# Hourglass Stealth Calculation Sheet

Technical calculation notebook for the blog-supporting analysis.

This first pass treats spacecraft emission as homogeneous across the surface and assumes perfect sun-axis alignment in all cases. Those are deliberate simplifications and should be read as conservative in favor of defense.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

import hourglass_stealth
from hourglass_stealth.constants import (
    EARTH_BOND_ALBEDO,
    EARTH_EFFECTIVE_IR_FLUX_W_M2,
    EARTH_RADIUS_M,
    EMISSIVITY_GRADES,
    MOON_BOND_ALBEDO,
    MOON_EFFECTIVE_IR_FLUX_W_M2,
    MOON_RADIUS_M,
    OPTICAL_ABSORPTION_GRADES,
    SOLAR_FLUX_1_AU_W_M2,
)
from hourglass_stealth.detector import detector_collecting_area_m2
from hourglass_stealth.emission import graybody_luminosity_w, wien_peak_um
from hourglass_stealth.environment import (
    body_albedo_flux_w_m2,
    body_ir_flux_w_m2,
    dwell_multiplier_from_extra_heat,
    environmental_heat_load_w,
)
from hourglass_stealth.heat_store import final_state_fits, heat_store_mass_kg, usable_energy_density_j_m3
from hourglass_stealth.optics import absorbed_solar_power, total_absorption_fraction
from hourglass_stealth.scenarios import default_heat_store_for_temperature, evaluate_scenario
from hourglass_stealth.tables import (
    emissivity_grade_table,
    heat_store_candidate_table,
    optical_absorption_grade_table,
)

OUTPUTS_DIR = Path('outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)

pd.set_option('display.float_format', lambda x: f'{x:,.6g}')
hourglass_stealth.__version__

'0.1.0'

## 0. Purpose and Governing Assumptions

This notebook is the technical calculation sheet for the hourglass stealth spacecraft concept.

Primary viability metrics:

1. Passive sealed dwell time, in years.
2. Detection distance, in Earth-Moon distances.

Out of scope in this pass:

- active cooling
- coolant ejection
- detailed detector engineering
- structural design
- tactical operations

Important limitation: the real observability problem is angle-dependent. This version intentionally collapses that into a homogeneous graybody treatment over the surface, which is conservative in favor of defense.

## 1. Design Rating Tables

In [2]:
optical_df = optical_absorption_grade_table()
emissivity_df = emissivity_grade_table()
heat_store_df = heat_store_candidate_table()

display(optical_df)
display(emissivity_df)
display(heat_store_df)

,grade,mirror_absorption_per_pass,lens_absorption_per_pass,interpretation
0,A,1e-05,0.0001,heroic / speculative
1,B,0.0001,0.001,aggressive advanced engineering
2,C,0.001,0.01,plausible-good but costly for stealth
3,D,0.01,0.1,poor for stealth


,grade,emissivity,interpretation
0,A,0.0001,heroic low-emissivity surface
1,B,0.001,aggressive engineered surface
2,C,0.01,good reflective surface
3,D,0.1,poor for stealth


,temperature_band_k,material_class,rho_initial_kg_m3,rho_final_kg_m3,usable_energy_j_kg,usable_energy_mj_kg,pressure_flag,notes
0,4-25,liquid hydrogen,71,65,"220,000",0.22,medium,Placeholder liquid-to-liquid deep-cryogenic ba...
1,25-90,liquid nitrogen,810,760,"120,000",0.12,low,Placeholder baseline around the 77 K regime.
2,90-140,liquid methane,420,370,"150,000",0.15,medium,Placeholder condensed-fluid baseline above nit...
3,140-260,liquid ammonia,680,610,"180,000",0.18,medium,Placeholder mid-cryogenic condensed-fluid base...
4,260-330,water,998,983,"200,000",0.2,low,Warm-case volumetric comparison baseline.


## 2. Architecture Comparison: Mirror vs Lens Absorption

In [3]:
aperture_area_m2 = 10.0
n_passes = 2
rows = []
for architecture in ('mirror', 'lens'):
    for grade, values in OPTICAL_ABSORPTION_GRADES.items():
        abs_per_pass = values[f'{architecture}_absorption_per_pass']
        total_abs = total_absorption_fraction(abs_per_pass, n_passes=n_passes)
        absorbed_per_m2 = absorbed_solar_power(SOLAR_FLUX_1_AU_W_M2, 1.0, abs_per_pass, n_passes=n_passes)
        absorbed_total = absorbed_solar_power(SOLAR_FLUX_1_AU_W_M2, aperture_area_m2, abs_per_pass, n_passes=n_passes)
        rows.append({
            'Architecture': architecture,
            'Grade': grade,
            'Absorption per pass': abs_per_pass,
            'Total absorption after 2 passes': total_abs,
            'Absorbed W per m^2 aperture': absorbed_per_m2,
            'Absorbed W for 10 m^2 aperture': absorbed_total,
            'Interpretation': values['interpretation'],
        })
architecture_absorption_df = pd.DataFrame(rows)
display(architecture_absorption_df)

,Architecture,Grade,Absorption per pass,Total absorption after 2 passes,Absorbed W per m^2 aperture,Absorbed W for 10 m^2 aperture,Interpretation
0,mirror,A,1e-05,1.99999e-05,0.0272199,0.272199,heroic / speculative
1,mirror,B,0.0001,0.00019999,0.272186,2.72186,aggressive advanced engineering
2,mirror,C,0.001,0.001999,2.72064,27.2064,plausible-good but costly for stealth
3,mirror,D,0.01,0.0199,27.0839,270.839,poor for stealth
4,lens,A,0.0001,0.00019999,0.272186,2.72186,heroic / speculative
5,lens,B,0.001,0.001999,2.72064,27.2064,aggressive advanced engineering
6,lens,C,0.01,0.0199,27.0839,270.839,plausible-good but costly for stealth
7,lens,D,0.1,0.19,258.59,"2,585.9",poor for stealth


## 3. Heat-Store Candidates Under Sealed-Volume Constraints

In [4]:
tank_volume_m3 = 10.0
sample_temperatures = {
    'liquid hydrogen': (20.0, 24.0),
    'liquid nitrogen': (77.0, 85.0),
    'liquid methane': (100.0, 120.0),
    'liquid ammonia': (180.0, 220.0),
    'water': (280.0, 320.0),
}
rows = []
for candidate in hourglass_stealth.scenarios.HEAT_STORE_DEFAULTS if False else []:
    pass
for candidate in heat_store_df.to_dict(orient='records'):
    t_initial, t_final = sample_temperatures[candidate['material_class']]
    mass_kg = heat_store_mass_kg(tank_volume_m3, candidate['rho_initial_kg_m3'])
    final_volume_ratio = candidate['rho_initial_kg_m3'] / candidate['rho_final_kg_m3']
    rows.append({
        'Heat store': candidate['material_class'],
        'State path': 'liquid -> liquid',
        'T_initial_K': t_initial,
        'T_final_K': t_final,
        'rho_initial_kg_m3': candidate['rho_initial_kg_m3'],
        'rho_final_kg_m3': candidate['rho_final_kg_m3'],
        'usable_energy_MJ_kg': candidate['usable_energy_mj_kg'],
        'usable_energy_MJ_m3': usable_energy_density_j_m3(candidate['rho_initial_kg_m3'], candidate['usable_energy_j_kg']) / 1e6,
        'final_volume_ratio': final_volume_ratio,
        'sealed_volume_valid': final_state_fits(mass_kg, candidate['rho_final_kg_m3'], tank_volume_m3),
        'pressure_flag': candidate['pressure_flag'],
        'notes': candidate['notes'],
    })
heat_store_candidates_df = pd.DataFrame(rows)
display(heat_store_candidates_df)

,Heat store,State path,T_initial_K,T_final_K,rho_initial_kg_m3,rho_final_kg_m3,usable_energy_MJ_kg,usable_energy_MJ_m3,final_volume_ratio,sealed_volume_valid,pressure_flag,notes
0,liquid hydrogen,liquid -> liquid,20,24,71,65,0.22,15.62,1.09231,False,medium,Placeholder liquid-to-liquid deep-cryogenic ba...
1,liquid nitrogen,liquid -> liquid,77,85,810,760,0.12,97.2,1.06579,False,low,Placeholder baseline around the 77 K regime.
2,liquid methane,liquid -> liquid,100,120,420,370,0.15,63,1.13514,False,medium,Placeholder condensed-fluid baseline above nit...
3,liquid ammonia,liquid -> liquid,180,220,680,610,0.18,122.4,1.11475,False,medium,Placeholder mid-cryogenic condensed-fluid base...
4,water,liquid -> liquid,280,320,998,983,0.2,199.6,1.01526,False,low,Warm-case volumetric comparison baseline.


## 4. Emissivity and Thermal Luminosity

In [5]:
emitting_area_m2 = 10.0
temperatures_k = [4, 10, 20, 50]
rows = []
for temperature_k in temperatures_k:
    for grade, values in EMISSIVITY_GRADES.items():
        rows.append({
            'Temperature K': temperature_k,
            'Wien peak um': wien_peak_um(temperature_k),
            'Emissivity grade': grade,
            'Emissivity': values['emissivity'],
            'Thermal luminosity W': graybody_luminosity_w(values['emissivity'], emitting_area_m2, temperature_k),
        })
thermal_luminosity_df = pd.DataFrame(rows)
display(thermal_luminosity_df)

,Temperature K,Wien peak um,Emissivity grade,Emissivity,Thermal luminosity W
0,4,724.5,A,0.0001,1.45162e-08
1,4,724.5,B,0.001,1.45162e-07
2,4,724.5,C,0.01,1.45162e-06
3,4,724.5,D,0.1,1.45162e-05
4,10,289.8,A,0.0001,5.67037e-07
5,10,289.8,B,0.001,5.67037e-06
6,10,289.8,C,0.01,5.67037e-05
7,10,289.8,D,0.1,0.000567037
8,20,144.9,A,0.0001,9.0726e-06
9,20,144.9,B,0.001,9.0726e-05


## 5. Detector Model and JWST-Scale Benchmark

In [6]:
baseline_diameter_m = 6.5
diameters_m = [1.0, 3.0, 6.5, 10.0, 30.0]
baseline_area = detector_collecting_area_m2(baseline_diameter_m)
detector_scaling_df = pd.DataFrame([
    {
        'Detector diameter m': diameter_m,
        'Collecting area m^2': detector_collecting_area_m2(diameter_m),
        'Detection range multiplier vs 6.5 m': diameter_m / baseline_diameter_m,
    }
    for diameter_m in diameters_m
])
display(detector_scaling_df)

,Detector diameter m,Collecting area m^2,Detection range multiplier vs 6.5 m
0,1,0.785398,0.153846
1,3,7.06858,0.461538
2,6.5,33.1831,1
3,10,78.5398,1.53846
4,30,706.858,4.61538


## 6. Single Baseline Scenario

In [7]:
baseline_result = evaluate_scenario(
    architecture='mirror',
    optical_grade='B',
    emissivity_grade='B',
    aperture_area_m2=10.0,
    emitting_area_m2=10.0,
    heat_store_volume_m3=10.0,
    spacecraft_temperature_k=20.0,
    detector_diameter_m=6.5,
    detector_throughput=0.3,
    integration_time_s=1000.0,
    required_signal_photons=25.0,
    survey_penalty_factor=1000.0,
)
baseline_scenario_df = pd.DataFrame([baseline_result])[['absorbed_solar_power_W', 'heat_store_energy_MJ', 'dwell_time_years', 'thermal_luminosity_W', 'wien_peak_um', 'pointed_detection_distance_EM', 'survey_detection_distance_EM', 'heat_store_material_class']]
display(baseline_scenario_df)

,absorbed_solar_power_W,heat_store_energy_MJ,dwell_time_years,thermal_luminosity_W,wien_peak_um,pointed_detection_distance_EM,survey_detection_distance_EM,heat_store_material_class
0,2.72186,156.2,1.81849,9.0726e-05,144.9,2.74327,0.0867497,liquid hydrogen


## 7. Scenario Grid: A/B/C Design Grades

In [8]:
architectures = ['mirror', 'lens']
optical_grades = ['A', 'B', 'C']
emissivity_grades = ['A', 'B', 'C']
temperatures_k = [4, 10, 20, 50]
heat_store_volumes_m3 = [1, 10, 100]
detector_diameters_m = [1, 6.5, 30]
rows = []
for architecture in architectures:
    for optical_grade in optical_grades:
        for emissivity_grade in emissivity_grades:
            for temperature_k in temperatures_k:
                for heat_store_volume_m3 in heat_store_volumes_m3:
                    for detector_diameter_m in detector_diameters_m:
                        rows.append(evaluate_scenario(
                            architecture=architecture,
                            optical_grade=optical_grade,
                            emissivity_grade=emissivity_grade,
                            aperture_area_m2=10.0,
                            emitting_area_m2=10.0,
                            heat_store_volume_m3=heat_store_volume_m3,
                            spacecraft_temperature_k=temperature_k,
                            detector_diameter_m=detector_diameter_m,
                            detector_throughput=0.3,
                            integration_time_s=1000.0,
                            required_signal_photons=25.0,
                            survey_penalty_factor=1000.0,
                        ))
scenario_grid_df = pd.DataFrame(rows)

architecture_penalty_df = scenario_grid_df.groupby('architecture')[['absorbed_solar_power_W', 'dwell_time_years', 'survey_detection_distance_EM']].mean().reset_index()
temperature_penalty_df = scenario_grid_df.groupby('wien_peak_um')[['dwell_time_years', 'thermal_luminosity_W', 'survey_detection_distance_EM']].mean().reset_index()
emissivity_penalty_df = scenario_grid_df.groupby('emissivity_grade')[['thermal_luminosity_W', 'survey_detection_distance_EM']].mean().reset_index()
detector_penalty_df = scenario_grid_df.groupby('detector_diameter_m')[['pointed_detection_distance_EM', 'survey_detection_distance_EM']].mean().reset_index()

display(architecture_penalty_df)
display(temperature_penalty_df)
display(emissivity_penalty_df)
display(detector_penalty_df)

,architecture,absorbed_solar_power_W,dwell_time_years,survey_detection_distance_EM
0,lens,100.256,5.74055,0.335951
1,mirror,10.0668,57.3985,0.335951


,wien_peak_um,dwell_time_years,thermal_luminosity_W,survey_detection_distance_EM
0,57.96,85.2022,0.0131127,0.984434
1,144.9,13.692,0.000335686,0.249044
2,289.8,13.692,2.09804e-05,0.0880505
3,724.5,13.692,5.37098e-07,0.0222752


,emissivity_grade,thermal_luminosity_W,survey_detection_distance_EM
0,A,9.10131e-05,0.0711646
1,B,0.000910131,0.225042
2,C,0.00910131,0.711646


,detector_diameter_m,pointed_detection_distance_EM,survey_detection_distance_EM
0,1,0.849896,0.0268761
1,6.5,5.52433,0.174695
2,30,25.4969,0.806282


## 8. Search for Viable Design Regions

In [9]:
min_dwell_years = 1.0
max_survey_detection_range_em = 1.0

def classify_case(row):
    dwell_ratio = row['dwell_time_years'] / min_dwell_years
    survey_ratio = row['survey_detection_distance_EM'] / max_survey_detection_range_em
    strong = dwell_ratio >= 1.0 and survey_ratio <= 1.0
    marginal = dwell_ratio >= 1/3 or survey_ratio <= 3.0
    if strong:
        return 'Strong'
    if marginal:
        return 'Marginal'
    return 'Dead'

viability_df = scenario_grid_df.copy()
viability_df['Temperature K'] = viability_df['wien_peak_um'].map(lambda peak: round(2898.0 / peak, 6))
viability_df['Verdict'] = viability_df.apply(classify_case, axis=1)
summary_cols = ['architecture', 'optical_grade', 'emissivity_grade', 'Temperature K', 'heat_store_energy_MJ', 'detector_diameter_m', 'dwell_time_years', 'survey_detection_distance_EM', 'Verdict']
display(viability_df[summary_cols].head(20))

top_dwell_df = viability_df[viability_df['survey_detection_distance_EM'] < 1.0].nlargest(10, 'dwell_time_years')[summary_cols]
top_stealth_df = viability_df[viability_df['dwell_time_years'] > 1.0].nsmallest(10, 'survey_detection_distance_EM')[summary_cols]
smallest_viable_df = viability_df[(viability_df['dwell_time_years'] >= 1.0) & (viability_df['survey_detection_distance_EM'] <= 1.0)].nsmallest(10, 'heat_store_energy_MJ')[summary_cols]

display(top_dwell_df)
display(top_stealth_df)
display(smallest_viable_df)

,architecture,optical_grade,emissivity_grade,Temperature K,heat_store_energy_MJ,detector_diameter_m,dwell_time_years,survey_detection_distance_EM,Verdict
0,mirror,A,A,4,15.62,1,1.81841,0.000377485,Strong
1,mirror,A,A,4,15.62,6.5,1.81841,0.00245365,Strong
2,mirror,A,A,4,15.62,30,1.81841,0.0113246,Strong
3,mirror,A,A,4,156.2,1,18.1841,0.000377485,Strong
4,mirror,A,A,4,156.2,6.5,18.1841,0.00245365,Strong
5,mirror,A,A,4,156.2,30,18.1841,0.0113246,Strong
6,mirror,A,A,4,"1,562",1,181.841,0.000377485,Strong
7,mirror,A,A,4,"1,562",6.5,181.841,0.00245365,Strong
8,mirror,A,A,4,"1,562",30,181.841,0.0113246,Strong
9,mirror,A,A,10,15.62,1,1.81841,0.00149214,Strong


,architecture,optical_grade,emissivity_grade,Temperature K,heat_store_energy_MJ,detector_diameter_m,dwell_time_years,survey_detection_distance_EM,Verdict
33,mirror,A,A,50,"9,720",1,"1,131.56",0.0166826,Strong
34,mirror,A,A,50,"9,720",6.5,"1,131.56",0.108437,Strong
35,mirror,A,A,50,"9,720",30,"1,131.56",0.500479,Strong
69,mirror,A,B,50,"9,720",1,"1,131.56",0.0527551,Strong
70,mirror,A,B,50,"9,720",6.5,"1,131.56",0.342908,Strong
105,mirror,A,C,50,"9,720",1,"1,131.56",0.166826,Strong
6,mirror,A,A,4,"1,562",1,181.841,0.000377485,Strong
7,mirror,A,A,4,"1,562",6.5,181.841,0.00245365,Strong
8,mirror,A,A,4,"1,562",30,181.841,0.0113246,Strong
15,mirror,A,A,10,"1,562",1,181.841,0.00149214,Strong


,architecture,optical_grade,emissivity_grade,Temperature K,heat_store_energy_MJ,detector_diameter_m,dwell_time_years,survey_detection_distance_EM,Verdict
0,mirror,A,A,4,15.62,1,1.81841,0.000377485,Strong
3,mirror,A,A,4,156.2,1,18.1841,0.000377485,Strong
6,mirror,A,A,4,"1,562",1,181.841,0.000377485,Strong
111,mirror,B,A,4,156.2,1,1.81849,0.000377485,Strong
114,mirror,B,A,4,"1,562",1,18.1849,0.000377485,Strong
222,mirror,C,A,4,"1,562",1,1.81931,0.000377485,Strong
327,lens,A,A,4,156.2,1,1.81849,0.000377485,Strong
330,lens,A,A,4,"1,562",1,18.1849,0.000377485,Strong
438,lens,B,A,4,"1,562",1,1.81931,0.000377485,Strong
36,mirror,A,B,4,15.62,1,1.81841,0.00119371,Strong


,architecture,optical_grade,emissivity_grade,Temperature K,heat_store_energy_MJ,detector_diameter_m,dwell_time_years,survey_detection_distance_EM,Verdict
0,mirror,A,A,4,15.62,1,1.81841,0.000377485,Strong
1,mirror,A,A,4,15.62,6.5,1.81841,0.00245365,Strong
2,mirror,A,A,4,15.62,30,1.81841,0.0113246,Strong
9,mirror,A,A,10,15.62,1,1.81841,0.00149214,Strong
10,mirror,A,A,10,15.62,6.5,1.81841,0.00969891,Strong
11,mirror,A,A,10,15.62,30,1.81841,0.0447642,Strong
18,mirror,A,A,20,15.62,1,1.81841,0.00422041,Strong
19,mirror,A,A,20,15.62,6.5,1.81841,0.0274327,Strong
20,mirror,A,A,20,15.62,30,1.81841,0.126612,Strong
36,mirror,A,B,4,15.62,1,1.81841,0.00119371,Strong


## 9. Optimized Candidate Cases

In [10]:
candidate_specs = [
    ('Mirror optimistic', dict(architecture='mirror', optical_grade='A', emissivity_grade='A', spacecraft_temperature_k=10.0, heat_store_volume_m3=100.0, detector_diameter_m=1.0)),
    ('Mirror baseline', dict(architecture='mirror', optical_grade='B', emissivity_grade='B', spacecraft_temperature_k=20.0, heat_store_volume_m3=10.0, detector_diameter_m=6.5)),
    ('Mirror pessimistic', dict(architecture='mirror', optical_grade='C', emissivity_grade='C', spacecraft_temperature_k=50.0, heat_store_volume_m3=1.0, detector_diameter_m=30.0)),
    ('Lens optimistic', dict(architecture='lens', optical_grade='A', emissivity_grade='A', spacecraft_temperature_k=10.0, heat_store_volume_m3=100.0, detector_diameter_m=1.0)),
    ('Lens baseline', dict(architecture='lens', optical_grade='B', emissivity_grade='B', spacecraft_temperature_k=20.0, heat_store_volume_m3=10.0, detector_diameter_m=6.5)),
]
rows = []
for scenario_name, overrides in candidate_specs:
    result = evaluate_scenario(
        aperture_area_m2=10.0,
        emitting_area_m2=10.0,
        detector_throughput=0.3,
        integration_time_s=1000.0,
        required_signal_photons=25.0,
        survey_penalty_factor=1000.0,
        **overrides,
    )
    rows.append({
        'Scenario': scenario_name,
        'Architecture': result['architecture'],
        'Optical grade': result['optical_grade'],
        'Emissivity grade': result['emissivity_grade'],
        'Temperature K': overrides['spacecraft_temperature_k'],
        'Aperture area m^2': 10.0,
        'Emitting area m^2': 10.0,
        'Heat-store volume m^3': overrides['heat_store_volume_m3'],
        'Detector diameter m': overrides['detector_diameter_m'],
        'Dwell years': result['dwell_time_years'],
        'Survey detection range EM': result['survey_detection_distance_EM'],
    })
optimized_cases_df = pd.DataFrame(rows)
display(optimized_cases_df)

,Scenario,Architecture,Optical grade,Emissivity grade,Temperature K,Aperture area m^2,Emitting area m^2,Heat-store volume m^3,Detector diameter m,Dwell years,Survey detection range EM
0,Mirror optimistic,mirror,A,A,10,10,10,100,1,181.841,0.00149214
1,Mirror baseline,mirror,B,B,20,10,10,10,6.5,1.81849,0.0867497
2,Mirror pessimistic,mirror,C,C,50,10,10,1,30,0.113212,5.00479
3,Lens optimistic,lens,A,A,10,10,10,100,1,18.1849,0.00149214
4,Lens baseline,lens,B,B,20,10,10,10,6.5,0.181931,0.0867497


## 10. Cislunar Proximity Penalties: Earth and Moon Heating

In [11]:
projected_area_m2 = 10.0
alpha_visible_planet = 1e-3
alpha_ir_planet = 1e-2
phase_factor = 0.5
baseline_heat_w = baseline_result['total_heat_load_W']

earth_distances_m = [1.1 * EARTH_RADIUS_M, 2 * EARTH_RADIUS_M, 5 * EARTH_RADIUS_M, 10 * EARTH_RADIUS_M, 20 * EARTH_RADIUS_M, 60 * EARTH_RADIUS_M]
moon_distances_m = [1.1 * MOON_RADIUS_M, 2 * MOON_RADIUS_M, 5 * MOON_RADIUS_M, 10 * MOON_RADIUS_M, 20 * MOON_RADIUS_M]

def heating_table(body_name, emit_flux, albedo, radius_m, distances_m):
    rows = []
    for distance_m in distances_m:
        ir_flux = body_ir_flux_w_m2(emit_flux, radius_m, distance_m)
        albedo_flux = body_albedo_flux_w_m2(SOLAR_FLUX_1_AU_W_M2, albedo, phase_factor, radius_m, distance_m)
        extra_heat_w = environmental_heat_load_w(projected_area_m2, alpha_visible_planet, alpha_ir_planet, albedo_flux, ir_flux)
        rows.append({
            f'Distance from {body_name} center m': distance_m,
            f'Distance in {body_name} radii': distance_m / radius_m,
            f'{body_name} IR flux W/m^2': ir_flux,
            f'{body_name} albedo flux W/m^2': albedo_flux,
            'Extra absorbed heat W': extra_heat_w,
            'Dwell multiplier': dwell_multiplier_from_extra_heat(baseline_heat_w, extra_heat_w),
        })
    return pd.DataFrame(rows)

earth_heating_df = heating_table('Earth', EARTH_EFFECTIVE_IR_FLUX_W_M2, EARTH_BOND_ALBEDO, EARTH_RADIUS_M, earth_distances_m)
moon_heating_df = heating_table('Moon', MOON_EFFECTIVE_IR_FLUX_W_M2, MOON_BOND_ALBEDO, MOON_RADIUS_M, moon_distances_m)
display(earth_heating_df)
display(moon_heating_df)

,Distance from Earth center m,Distance in Earth radii,Earth IR flux W/m^2,Earth albedo flux W/m^2,Extra absorbed heat W,Dwell multiplier
0,7.0081e+06,1.1,197.521,168.719,21.4393,0.112655
1,1.2742e+07,2,59.75,51.0375,6.48537,0.295622
2,3.1855e+07,5,9.56,8.166,1.03766,0.723992
3,6.371e+07,10,2.39,2.0415,0.259415,0.912985
4,1.2742e+08,20,0.5975,0.510375,0.0648538,0.976728
5,3.8226e+08,60,0.0663889,0.0567083,0.00720597,0.99736


,Distance from Moon center m,Distance in Moon radii,Moon IR flux W/m^2,Moon albedo flux W/m^2,Extra absorbed heat W,Dwell multiplier
0,1.91114e+06,1.1,198.347,67.4876,20.5096,0.117163
1,3.4748e+06,2,60,20.415,6.20415,0.304936
2,8.687e+06,5,9.6,3.2664,0.992664,0.732762
3,1.7374e+07,10,2.4,0.8166,0.248166,0.916443
4,3.4748e+07,20,0.6,0.20415,0.0620415,0.977714


## 11. Operational Envelope in Cislunar Space

In [12]:
regions = [
    ('Near Earth orbit', 1.2 * EARTH_RADIUS_M, 60 * MOON_RADIUS_M),
    ('High Earth orbit', 10 * EARTH_RADIUS_M, 60 * MOON_RADIUS_M),
    ('Mid-cislunar', 30 * EARTH_RADIUS_M, 80 * MOON_RADIUS_M),
    ('Near lunar space', 60 * EARTH_RADIUS_M, 2 * MOON_RADIUS_M),
    ('Earth-Moon L1/L2 vicinity', 50 * EARTH_RADIUS_M, 15 * MOON_RADIUS_M),
    ('Deep cislunar away from both bodies', 100 * EARTH_RADIUS_M, 50 * MOON_RADIUS_M),
]
rows = []
for region, earth_distance_m, moon_distance_m in regions:
    earth_ir = body_ir_flux_w_m2(EARTH_EFFECTIVE_IR_FLUX_W_M2, EARTH_RADIUS_M, earth_distance_m)
    earth_albedo = body_albedo_flux_w_m2(SOLAR_FLUX_1_AU_W_M2, EARTH_BOND_ALBEDO, phase_factor, EARTH_RADIUS_M, earth_distance_m)
    moon_ir = body_ir_flux_w_m2(MOON_EFFECTIVE_IR_FLUX_W_M2, MOON_RADIUS_M, moon_distance_m)
    moon_albedo = body_albedo_flux_w_m2(SOLAR_FLUX_1_AU_W_M2, MOON_BOND_ALBEDO, phase_factor, MOON_RADIUS_M, moon_distance_m)
    earth_heat = environmental_heat_load_w(projected_area_m2, alpha_visible_planet, alpha_ir_planet, earth_albedo, earth_ir)
    moon_heat = environmental_heat_load_w(projected_area_m2, alpha_visible_planet, alpha_ir_planet, moon_albedo, moon_ir)
    total_extra_heat = earth_heat + moon_heat
    regional_case = evaluate_scenario(
        architecture='mirror',
        optical_grade='B',
        emissivity_grade='B',
        aperture_area_m2=10.0,
        emitting_area_m2=10.0,
        heat_store_volume_m3=10.0,
        spacecraft_temperature_k=20.0,
        detector_diameter_m=6.5,
        detector_throughput=0.3,
        integration_time_s=1000.0,
        required_signal_photons=25.0,
        survey_penalty_factor=1000.0,
        earth_heat_w=earth_heat,
        moon_heat_w=moon_heat,
    )
    rows.append({
        'Region': region,
        'Earth distance m': earth_distance_m,
        'Moon distance m': moon_distance_m,
        'Extra heat W': total_extra_heat,
        'Dwell years': regional_case['dwell_time_years'],
        'Survey range EM': regional_case['survey_detection_distance_EM'],
        'Viability': 'Yes' if regional_case['dwell_time_years'] >= 1.0 and regional_case['survey_detection_distance_EM'] <= 1.0 else 'No',
    })
operational_envelope_df = pd.DataFrame(rows)
display(operational_envelope_df)

,Region,Earth distance m,Moon distance m,Extra heat W,Dwell years,Survey range EM,Viability
0,Near Earth orbit,7.6452e+06,1.04244e+08,18.0218,0.238611,0.0867497,No
1,High Earth orbit,6.371e+07,1.04244e+08,0.266309,1.65642,0.0867497,Yes
2,Mid-cislunar,1.9113e+08,1.38992e+08,0.0327015,1.7969,0.0867497,Yes
3,Near lunar space,3.8226e+08,3.4748e+06,6.21136,0.554076,0.0867497,No
4,Earth-Moon L1/L2 vicinity,3.1855e+08,2.6061e+07,0.120673,1.74129,0.0867497,Yes
5,Deep cislunar away from both bodies,6.371e+08,8.687e+07,0.0125208,1.81016,0.0867497,Yes


## 12. Blog-Ready Table Exports

In [13]:
exports = {
    'optical_absorption_grades.csv': optical_df,
    'emissivity_grades.csv': emissivity_df,
    'architecture_absorption_comparison.csv': architecture_absorption_df,
    'thermal_luminosity_table.csv': thermal_luminosity_df,
    'baseline_scenario.csv': baseline_scenario_df,
    'scenario_search.csv': viability_df,
    'optimized_candidate_cases.csv': optimized_cases_df,
    'earth_heating_penalty.csv': earth_heating_df,
    'moon_heating_penalty.csv': moon_heating_df,
}
for filename, dataframe in exports.items():
    dataframe.to_csv(OUTPUTS_DIR / filename, index=False)
sorted(path.name for path in OUTPUTS_DIR.glob('*.csv'))

['architecture_absorption_comparison.csv',
 'baseline_scenario.csv',
 'earth_heating_penalty.csv',
 'emissivity_grades.csv',
 'moon_heating_penalty.csv',
 'optical_absorption_grades.csv',
 'optimized_candidate_cases.csv',
 'scenario_search.csv',
 'thermal_luminosity_table.csv']